## Extract data

In [35]:
import pandas as pd
import numpy as np

In [36]:
# # Ensure the required library is installed 
# (Can we do it (openpyxl)? Will it be a problem?)
# %pip install openpyxl

# Read the Excel file
file_path = "bitre_fatalities_dec2024.xlsx"
df = pd.read_excel(file_path, sheet_name="BITRE_Fatality", skiprows=4) # Can be improved


In [37]:
# Clean the columnnames
# Remove leading and trailing whitespace, convert to lowercase, and replace spaces with underscores

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

# change sa4_name_2021 to sa4_name, national_lga_name_2021 to lga_name
df.rename(columns={
    'national_lga_name_2021': 'lga_name',
    'national_road_type': 'road_type'
}, inplace=True)
df.columns


Index(['crash_id', 'state', 'month', 'year', 'dayweek', 'time', 'crash_type',
       'bus_involvement', 'heavy_rigid_truck_involvement',
       'articulated_truck_involvement', 'speed_limit', 'road_user', 'gender',
       'age', 'national_remoteness_areas', 'sa4_name_2021', 'lga_name',
       'road_type', 'christmas_period', 'easter_period', 'age_group',
       'day_of_week', 'time_of_day'],
      dtype='object')

In [38]:
# Add a serial number for each person killed in the accident
df['victim_number'] = df.groupby('crash_id').cumcount() + 1

# move the victim_number column to the front
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index('victim_number')))
df = df[cols]
print(df.head())



   crash_id  victim_number state  month  year dayweek      time crash_type  \
0  20241115              1   NSW     12  2024  Friday  04:00:00     Single   
1  20241125              1   NSW     12  2024  Friday  06:15:00     Single   
2  20246013              1   Tas     12  2024  Friday  09:43:00   Multiple   
3  20241002              1   NSW     12  2024  Friday  10:35:00   Multiple   
4  20242261              1   Vic     12  2024  Friday  11:30:00   Multiple   

  bus_involvement heavy_rigid_truck_involvement  ... age  \
0              No                            No  ...  74   
1              No                            No  ...  19   
2              No                            No  ...  33   
3              No                            No  ...  32   
4              -9                            -9  ...  62   

  national_remoteness_areas                           sa4_name_2021  \
0  Inner Regional Australia                                Riverina   
1  Inner Regional Australia 

In [39]:
# drop 'age', 'time', 'road_user', 'gender', 'national_remoteness_areas', 'sa4_name_2021', 'day_of_week'
df.drop(columns=['age', 'road_user', 'gender', 'national_remoteness_areas', 'sa4_name_2021', 'day_of_week'], inplace=True)

In [40]:
# Save the cleaned DataFrame to a new Excel file
# output_file_path = "bitre_fatalities_cleaned.xlsx"
# df.to_excel(output_file_path, index=False)
# print(f"Cleaned data saved to {output_file_path}")


## Data transformation

Data transformation to apply:

1. dayweek: Drop
2. time: Categorise into rush time and usual time:
    Rush hours: 
    Morning Peak:
    Typically between 7 am and 9 am, as commuters head to work or school. 

    Evening Peak:
    Typically between 4 pm and 6 pm, as commuters travel home from work or school. 

    Not holiday, not weekend

Can be improved according to the state, city and so on

3. bus_involvement, heavy_rigid_truck_involvement, articulated_truck_involvement - treat -9 missing values
4. speed_limit: Categorise as follows:
    For all except NT:
        0-40 - low
        41-50 - med
        51-80 - high
        81 - inf - very high
    
    For NT:
        0-40 - low
        41-60 - med
        61-80 - high
        81 - inf - very high

    treat -9 as missing value
5. road_user:
    treat Other/-9, Unknown - as missing value

6. gender:
    treat -9 - as missing value

7. age: drop

8. national_remoteness_areas:
    treat Unknown - as missing value

9. sa4_name_2021:
    treat Unknown, Blank - as missing value

10. national_lga_name_2021:
    treat Unknown, Blank - as missing value

11. national_road_type:
    treat Undetermined - as missing value

12. christmas_period, easter_period:
    transform into is_holiday

13. age_group:
    treat -9 - as missing value

14. day_of_week:
    treat Unknown - as missing value

15. time_of_day:
    treat Unknown - as missing value

In [41]:
# convert the 'time' column to datetime format
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.time

# categorize time of day by rush hous:
# For all holiday == "No", day_of_week == "Weekday" set rush: 
# 07:00:00 - 09:00:00 = "Rush"
# 16:00:00 - 18:00:00 = "Rush"
Weekday = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

conditions = [
    (df['christmas_period'] == "No") & (df['easter_period'] == "No") & (df['dayweek'].isin(Weekday)) & (
    ((df['time'] >= pd.to_datetime("07:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("09:00:00", format='%H:%M:%S').time())) |
    ((df['time'] >= pd.to_datetime("16:00:00", format='%H:%M:%S').time()) & (df['time'] <= pd.to_datetime("18:00:00", format='%H:%M:%S').time()))
    ),
    (df['time'].isna())
]

choices = ["Rush", np.nan]

df['time_cat'] = np.select(conditions, choices, default="Not Rush")

# change 'nan' value to np.nan
df['time_cat'] = df['time_cat'].replace('nan', np.nan)            # КОСТЫЛЬ

print(df['time_cat'].unique())


['Not Rush' 'Rush' nan]


In [42]:
# set  NaN values for all 'Other/-9', '-9', 'Unknown', 'Undetermined' in all columns
nan_values = ['Other/-9', '-9', 'Unknown', 'Undetermined', -9]
df.replace(nan_values, np.nan, inplace=True)
print(df.head())

   crash_id  victim_number state  month  year dayweek      time crash_type  \
0  20241115              1   NSW     12  2024  Friday  04:00:00     Single   
1  20241125              1   NSW     12  2024  Friday  06:15:00     Single   
2  20246013              1   Tas     12  2024  Friday  09:43:00   Multiple   
3  20241002              1   NSW     12  2024  Friday  10:35:00   Multiple   
4  20242261              1   Vic     12  2024  Friday  11:30:00   Multiple   

  bus_involvement heavy_rigid_truck_involvement articulated_truck_involvement  \
0              No                            No                            No   
1              No                            No                            No   
2              No                            No                            No   
3              No                            No                            No   
4             NaN                           NaN                           NaN   

  speed_limit           lga_name            

In [43]:
# check for right data type 
try :
    df['speed_limit'] = df['speed_limit'].astype(float)
except ValueError:
    # If conversion to int fails, print values that cannot be converted
    print("Values that cannot be converted to int:")
    print(df[~df['speed_limit'].apply(lambda x: isinstance(x, int) or pd.isna(x))]['speed_limit'].unique())



Values that cannot be converted to int:
['<40']


<40 means less than 40, which is 'low' speed category
just set this value to 40

In [44]:
# set '<40' speed_limit to 40
df['speed_limit'] = df['speed_limit'].replace('<40', 40)

In [45]:
# speed_limit: Categorise as follows:
#     For all except NT:
#         0-40 - low
#         41-50 - med
#         51-80 - high
#         81 - inf - very high
    
#     For NT:
#         0-40 - low
#         41-60 - med
#         61-80 - high
#         81 - inf - very high

df['speed_limit'] = np.where(
    df['state'] != "NT",
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 50),
            (df['speed_limit'] >= 51) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    ),
    np.select(
        [
            (df['speed_limit'] > 0) & (df['speed_limit'] <= 40),
            (df['speed_limit'] >= 41) & (df['speed_limit'] <= 60),
            (df['speed_limit'] >= 61) & (df['speed_limit'] <= 80),
            (df['speed_limit'] > 80)
        ],
        ['Low', 'Med', 'High', 'Very High'],
        default=np.nan
    )
)

# change 'nan' value to np.nan
df['speed_limit'] = df['speed_limit'].replace('nan', np.nan)            # КОСТЫЛЬ
# check if there are any NaN values in speed_limit
print(df['speed_limit'].isna().sum())


1485


In [46]:
print(df['speed_limit'].unique())

['Very High' 'High' 'Med' nan 'Low']


In [47]:
# create holiday column with values: Christmas, Easter, NoHoliday
conditions = [
    (df['christmas_period'] == "Yes"),
    (df['easter_period'] == "Yes"),
    (df['christmas_period'] == "No") & (df['easter_period'] == "No")
]
choices = ["Christmas", "Easter", "NoHoliday"]
df['holiday'] = np.select(conditions, choices, default=np.nan)

In [48]:
# create vehicle_type column with values: bus, heavy_truck, articulated_truck, noHeavyVehicle
conditions = [
    (df['bus_involvement'] == "Yes") | (df['heavy_rigid_truck_involvement'] == "Yes") | (df['articulated_truck_involvement'] == "Yes"),
    (df['bus_involvement'] == "No") & (df['heavy_rigid_truck_involvement'] == "No") & (df['articulated_truck_involvement'] == "No")
]

choices = ["Heavy Vehicle Involved", "No Heavy Vehicle Involved"]

df['vehicle_type_involved'] = np.select(conditions, choices, default=np.nan)
# change 'nan' value to np.nan
df['vehicle_type_involved'] = df['vehicle_type_involved'].replace('nan', np.nan)            # КОСТЫЛЬ

In [49]:
# Save the cleaned DataFrame to a new Excel file
output_file_path = "bitre_fatalities_cleaned2.xlsx"
df.to_excel(output_file_path, index=False)
print(f"Cleaned data saved to {output_file_path}")

Cleaned data saved to bitre_fatalities_cleaned2.xlsx


## Population table

you are required to utilise at least one of the following datasets: Dwelling Count Data or Population Data. You may choose to incorporate both of these additional datasets if desired.

In [50]:
file_path = "Population.xlsx"
population = pd.read_excel(file_path, sheet_name="Table 1", skiprows=5) # Can be improved
print(population.head())

  Unnamed: 0             Unnamed: 1   2001   2002   2003   2004   2005   2006  \
0   LGA code  Local Government Area    no.    no.    no.    no.    no.    no.   
1      10050                 Albury  45265  45816  46180  46505  47004  47566   
2      10180               Armidale  27906  27774  27610  27410  27350  27377   
3      10250                Ballina  37856  38417  38870  39120  39305  39537   
4      10300              Balranald   2751   2703   2661   2596   2545   2507   

    2007   2008  ...   2014   2015   2016   2017   2018   2019   2020   2021  \
0    no.    no.  ...    no.    no.    no.    no.    no.    no.    no.    no.   
1  48140  48518  ...  50990  51486  52171  53056  53922  54657  55466  56067   
2  27468  27788  ...  29015  29160  29310  29519  29631  29701  29600  29332   
3  39824  40020  ...  41881  42336  42993  43652  44385  44997  45663  46196   
4   2473   2433  ...   2376   2364   2330   2338   2308   2287   2257   2208   

    2022   2023  
0    no.    no

In [51]:
# Clean the columnnames
year_colnames = population.columns[2:].tolist()  # Get the first row for year column names
lga_colnames = population.iloc[0, :2].tolist()  # Get the first two columns for LGA names

# change 'local_government_area' to 'lga_name'
lga_colnames[1] = 'lga_name'


for i in range(len(lga_colnames)):
    lga_colnames[i] = lga_colnames[i].strip().lower().replace(' ', '_').replace('/', '_')
population.columns = lga_colnames + year_colnames  # Combine the two lists
population = population[1:-2].reset_index(drop=True)  # Skip the first row and the last 2 rows with '© Commonwealth of Australia' and Total Australia
population.head()
population.tail()

,lga_code,lga_name,2001,2002,2003,2004,2005,2006,2007,2008,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
542,74660,West Arnhem,6241,6198,6161,6182,6304,6452,6505,6679,...,7157,7060,6941,6943,6985,7059,7130,7182,7258,7407
543,74680,West Daly,2543,2612,2674,2752,2864,2986,3023,3151,...,3587,3598,3601,3536,3476,3427,3422,3422,3434,3426
544,79399,Unincorporated NT,7027,7072,7058,7128,7447,7664,7879,7995,...,7970,7112,7065,7034,7023,7102,7263,7420,7571,7713
545,89399,Unincorporated ACT,321538,324627,327357,328940,331399,335170,342644,348368,...,388799,395813,403104,415046,426081,435730,444903,452508,456915,466566
546,99399,Unincorp. Other Territories,542,464,441,428,413,386,370,370,...,361,367,2159,2243,2324,2382,2437,2530,2518,2516


In [52]:
# Transform population into long format
population_long = population.melt(
    id_vars = ["lga_code", "lga_name"],
    var_name = "year",
    value_name = "population"
)
population_long




,lga_code,lga_name,year,population
0,10050,Albury,2001,45265
1,10180,Armidale,2001,27906
2,10250,Ballina,2001,37856
3,10300,Balranald,2001,2751
4,10470,Bathurst,2001,35504
...,...,...,...,...
12576,74660,West Arnhem,2023,7407
12577,74680,West Daly,2023,3426
12578,79399,Unincorporated NT,2023,7713
12579,89399,Unincorporated ACT,2023,466566


## Designing dimention tables

In [53]:
print(df.columns)

Index(['crash_id', 'victim_number', 'state', 'month', 'year', 'dayweek',
       'time', 'crash_type', 'bus_involvement',
       'heavy_rigid_truck_involvement', 'articulated_truck_involvement',
       'speed_limit', 'lga_name', 'road_type', 'christmas_period',
       'easter_period', 'age_group', 'time_of_day', 'time_cat', 'holiday',
       'vehicle_type_involved'],
      dtype='object')


In [54]:
# create location_dimension table
location_dim = df[['state', 'lga_name']].drop_duplicates()

location_dim

# add lga_code to location_dim from population_long
location_dim = location_dim.merge(population_long[['lga_name', 'lga_code']].drop_duplicates(), on='lga_name', how='left')

# move lga_code to the front
location_dim = location_dim[['lga_code', 'state', 'lga_name']]

# check if there are lga_name != NaN and lge_code = NaN
print(location_dim[location_dim['lga_name'].notna() & location_dim['lga_code'].isna()]) # should be empty. Need to fix it   

    lga_code state                               lga_name
3        NaN   NSW                      Armidale Regional
21       NaN   NSW                   Mid-Western Regional
42       NaN   NSW                          Central Coast
43       NaN   NSW                      Tamworth Regional
51       NaN   NSW                         Dubbo Regional
93       NaN   NSW           Queanbeyan-Palerang Regional
94       NaN   NSW                  Snowy Monaro Regional
111      NaN   NSW                           Campbelltown
153      NaN   Vic                               Moreland
171      NaN   NSW                         Unincorporated
179      NaN   NSW                      Bathurst Regional
200      NaN   Tas                            Break O'Day
229      NaN   NSW                                Bayside
469      NaN    SA  Anangu Pitjantjatjara Yunkunytjatjara
480      NaN   NSW                               Nambucca


In [55]:
# find lga_code for NaN rows for lga_name using first word
location_dim[location_dim['lga_name'].notna() & location_dim['lga_code'].isna()]

def match_by_first_word(row):
    # Only try to match if lga_code is missing and lga_name exists
    if pd.isna(row['lga_code']) and pd.notna(row['lga_name']):
        first_word = row['lga_name'].split()[0]
        # Try to find match in population_long
        match = population_long[population_long['lga_name'].str.contains(rf'\b{first_word}\b', case=False, na=False)]
        if not match.empty:
            return match['lga_code'].values[0]
    return row['lga_code']

# Apply the function row-wise
location_dim['lga_code'] = location_dim.apply(match_by_first_word, axis=1)
# make lga_code to int, ignore NaN
location_dim['lga_code'] = location_dim['lga_code'].astype(pd.Int64Dtype())
location_dim[location_dim['lga_code'].isna()]


,lga_code,state,lga_name
4,<NA>,Vic,NaN
7,<NA>,WA,NaN
23,<NA>,NT,NaN
96,<NA>,ACT,NaN
153,<NA>,Vic,Moreland
451,<NA>,Tas,NaN
506,<NA>,Qld,NaN
513,<NA>,NSW,NaN
517,<NA>,SA,NaN


In [56]:
# drop lag_code == NaN
location_dim.dropna(subset=['lga_code'], inplace=True)
# check if there are any NaN values in lga_code
print(location_dim['lga_code'].isna().sum())

0


In [57]:
# create date_dimension table
date_dim = df[['year', 'month']].drop_duplicates()
date_dim['dateID'] = df['year'].astype(str) + df['month'].astype(str)
date_dim = date_dim[['dateID', 'year', 'month']]
date_dim

# check if there are any NaN values in dateID
print(date_dim['dateID'].isna().sum())

0


In [58]:
# Create rush_dim table
rush_dim = df[['time_cat']].drop_duplicates()
rush_dim['rushID'] = range(1, len(rush_dim) + 1)
rush_dim = rush_dim[['rushID', 'time_cat']]
rush_dim.dropna(subset=['time_cat'], inplace=True)
rush_dim

,rushID,time_cat
0,1,Not Rush
7,2,Rush


In [59]:
# Create age_dim table
age_dim = df[['age_group']].drop_duplicates()

# sort by age_group
age_dim = age_dim.sort_values(by='age_group')

age_dim['ageID'] = range(1, len(age_dim) + 1)
age_dim = age_dim[['ageID', 'age_group']]
age_dim.dropna(subset=['age_group'], inplace=True)
age_dim


,ageID,age_group
20,1,0_to_16
1,2,17_to_25
2,3,26_to_39
4,4,40_to_64
0,5,65_to_74
18,6,75_or_older


In [60]:
# create daytime dimension table
daytime_dim = df[['time_of_day']].drop_duplicates()
daytime_dim['daytimeID'] = range(1, len(daytime_dim) + 1)
daytime_dim = daytime_dim[['daytimeID', 'time_of_day']]
daytime_dim.dropna(subset=['time_of_day'], inplace=True)
daytime_dim

,daytimeID,time_of_day
0,1,Night
1,2,Day


In [61]:
# create road_type dimension table
road_type_dim = df[['road_type']].drop_duplicates()
road_type_dim['road_typeID'] = range(1, len(road_type_dim) + 1)
road_type_dim = road_type_dim[['road_typeID', 'road_type']]
road_type_dim.dropna(subset=['road_type'], inplace=True)
road_type_dim

,road_typeID,road_type
0,1,Arterial Road
1,2,Local Road
3,3,National or State Highway
6,5,Sub-arterial Road
17,6,Collector Road
154,7,Pedestrian Thoroughfare
283,8,Access road
970,9,Busway


In [62]:
# create speed_limit dimension table
speed_limit_dim = df[['speed_limit']].drop_duplicates().dropna()
speed_limit_dim['speed_limitID'] = range(1, len(speed_limit_dim) + 1)
speed_limit_dim = speed_limit_dim[['speed_limitID', 'speed_limit']]
speed_limit_dim.dropna(subset=['speed_limit'], inplace=True)
speed_limit_dim

,speed_limitID,speed_limit
0,1,Very High
1,2,High
2,3,Med
25,4,Low


In [63]:
# create holiday dimension table
holiday_dim = df[['holiday']].drop_duplicates().sort_values(by='holiday')
holiday_dim['holidayID'] = range(1, len(holiday_dim) + 1)
holiday_dim = holiday_dim[['holidayID', 'holiday']]
holiday_dim.dropna(subset=['holiday'], inplace=True)
holiday_dim

,holidayID,holiday
0,1,Christmas
892,2,Easter
1,3,NoHoliday


In [64]:
# create vehicle_type dimension table
vehicle_type_dim = df[['vehicle_type_involved']].drop_duplicates().dropna()
vehicle_type_dim['vehicle_typeID'] = range(1, len(vehicle_type_dim) + 1)
vehicle_type_dim = vehicle_type_dim[['vehicle_typeID', 'vehicle_type_involved']]
vehicle_type_dim.dropna(subset=['vehicle_type_involved'], inplace=True)
vehicle_type_dim

,vehicle_typeID,vehicle_type_involved
0,1,No Heavy Vehicle Involved
13,2,Heavy Vehicle Involved


In [69]:
fatalities_df = df.copy()
fatalities_df = df.merge(location_dim, on=['state', 'lga_name'], how='left')
fatalities_df = df.merge(date_dim, on=['year', 'month'], how='left')
fatalities_df = df.merge(rush_dim, on=['time_cat'], how='left')
fatalities_df = df.merge(age_dim, on=['age_group'], how='left')
fatalities_df = df.merge(daytime_dim, on=['time_of_day'], how='left')
fatalities_df = df.merge(road_type_dim, on=['road_type'], how='left')
fatalities_df = df.merge(speed_limit_dim, on=['speed_limit'], how='left')
fatalities_df = df.merge(holiday_dim, on=['holiday'], how='left')
fatalities_df = df.merge(vehicle_type_dim, on=['vehicle_type_involved'], how='left')

# create fact table
#dfs = [location_dim, date_dim, rush_dim, age_dim, daytime_dim, road_type_dim, speed_limit_dim, holiday_dim, vehicle_type_dim]
#for df in dfs:
#    fatalities_df = fatalities_df.merge(df, on=df.columns[1])
# Here we should calculate measures

# after that we can drop columns from the fact table
# fatalities_df = fatalities_df[['crash_id', 'victim_number', 'lga_code', 'dateID', 'rushID', 'ageID', 'daytimeID', 'road_typeID', 'speed_limitID', 'holidayID', 'vehicle_typeID', OUR_MEASURES]]

In [70]:
# dimensions (rows*col) of fatalities_df
print(fatalities_df)


       crash_id  victim_number state  month  year    dayweek      time  \
0      20241115              1   NSW     12  2024     Friday  04:00:00   
1      20241125              1   NSW     12  2024     Friday  06:15:00   
2      20246013              1   Tas     12  2024     Friday  09:43:00   
3      20241002              1   NSW     12  2024     Friday  10:35:00   
4      20242261              1   Vic     12  2024     Friday  11:30:00   
...         ...            ...   ...    ...   ...        ...       ...   
56869  19896006              3   Tas      1  1989  Wednesday  20:20:00   
56870  19896006              4   Tas      1  1989  Wednesday  20:20:00   
56871  19896006              5   Tas      1  1989  Wednesday  20:20:00   
56872  19896006              6   Tas      1  1989  Wednesday  20:20:00   
56873  19895133              1    WA      1  1989  Wednesday  21:00:00   

      crash_type bus_involvement heavy_rigid_truck_involvement  ...  \
0         Single              No        

In [67]:
# Это код из третьей лабы, который нужно адаптировать под наши данные
# Тут мерджим таблицы, которые мы получили выше, с таблицей transactions
# Потом считаем Measures, которые нам нужны
# Потом создаем таблицу sales_df, которая будет фактом
# И вот потом уже нужно будет применять алгоритмы
# проблема, что после мерджа у меня получилось 11 млн строк. Мерджил 3.5 минут

# sales_df = transactions.copy()
# dfs = [customer_df, product_df, location_df, date_df]
# for df in dfs:
#     sales_df = sales_df.merge(df, on = df.columns[1])
# sales_df["TotalPrice"] = round(sales_df["Unit_Price"] * sales_df["Quantity"],2)
# sales_df["ActualPrice"] = round(sales_df["TotalPrice"] * (1 - sales_df["Discount"] /100), 2)
# sales_df["SalesID"] = range(1, len(sales_df) +1)
# sales_df = sales_df[["SalesID", "ProductID", "CustomerID", "DateID", "LocationID", "Quantity", "TotalPrice", "Discount", "ActualPrice"]]
# sales_df

In [68]:
# # convert the DataFrame to a CSV file
# table_names = ["location_dim", "date_dim", "rush_dim", "age_dim", "daytime_dim", "road_type_dim", "speed_limit_dim", "holiday_dim", "vehicle_type_dim", "fatalities_df"]
# dfs.append(fatalities_df)
# for i in range(table_names):
#     dfs[i].to_csv(table_names[i] + ".csv", index=False)
#     print(f"Table {table_names[i]} saved to {table_names[i]}.csv")

In [72]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(df[['state','month','year','dayweek','crash_type','speed_limit','lga_name','road_type','holiday','age_group','time_of_day','time_cat','vehicle_type_involved']])

encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['state','month','year','dayweek','crash_type','speed_limit','lga_name','road_type','holiday','age_group','time_of_day','time_cat','vehicle_type_involved']))
print(encoded_df)

       state_ACT  state_NSW  state_NT  state_Qld  state_SA  state_Tas  \
0            0.0        1.0       0.0        0.0       0.0        0.0   
1            0.0        1.0       0.0        0.0       0.0        0.0   
2            0.0        0.0       0.0        0.0       0.0        1.0   
3            0.0        1.0       0.0        0.0       0.0        0.0   
4            0.0        0.0       0.0        0.0       0.0        0.0   
...          ...        ...       ...        ...       ...        ...   
56869        0.0        0.0       0.0        0.0       0.0        1.0   
56870        0.0        0.0       0.0        0.0       0.0        1.0   
56871        0.0        0.0       0.0        0.0       0.0        1.0   
56872        0.0        0.0       0.0        0.0       0.0        1.0   
56873        0.0        0.0       0.0        0.0       0.0        0.0   

       state_Vic  state_WA  month_1  month_2  ...  age_group_nan  \
0            0.0       0.0      0.0      0.0  ...      

d:\Anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [ ]:
#!pip install mlxtend
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.frequent_patterns import fpgrowth, association_rules

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB 5.2 MB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.4 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------  1.3/1.4 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 8.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   -- ------------------------------------- 0.6/11.1 MB 12.2 MB/s eta 0:00:01
   --- ------------------------------------ 1.1/11.1 MB 11.7 MB/s eta 0:00:01
   ------ --------------------------------- 1.7/11.1 MB 12.1 MB/s eta 0:00:01
   ------- -------------------------------- 2.2/11.1 MB 11.5 MB/s eta 0:00:01
   --------- ------------------------------ 2.8/11.1 MB 11.8 MB/s eta 0:00:01
   ----------- ---------------------------- 3.3/11.1 MB 11.7 MB/s eta 0:00:01
   ------------- -------------------------- 3.9/11.1 MB 11.8 MB/s eta 0:00:01
   --------

In [75]:
min_support = 0.05  


frequent_itemsets_ap = apriori(encoded_df, min_support=min_support, use_colnames=True)

print("Combos")
print(frequent_itemsets_ap.head())


min_confidence = 0.3
rules = association_rules(frequent_itemsets_ap, metric="confidence", min_threshold=min_confidence)

print("\n(Apriori):")
print(rules.head())

d:\Anaconda\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Combos
    support     itemsets
0  0.304638  (state_NSW)
1  0.201146  (state_Qld)
2  0.085329   (state_SA)
3  0.218659  (state_Vic)
4  0.120301   (state_WA)

(Apriori):
             antecedents            consequents  antecedent support  \
0       (dayweek_Friday)            (state_NSW)            0.164293   
1            (state_NSW)  (crash_type_Multiple)            0.304638   
2  (crash_type_Multiple)            (state_NSW)            0.443823   
3            (state_NSW)    (crash_type_Single)            0.304638   
4     (speed_limit_High)            (state_NSW)            0.421985   

   consequent support   support  confidence      lift  representativity  \
0            0.304638  0.050128    0.305116  1.001567               1.0   
1            0.443823  0.141857    0.465659  1.049198               1.0   
2            0.304638  0.141857    0.319626  1.049198               1.0   
3            0.556177  0.162781    0.534341  0.960740               1.0   
4            0.304638  0.1373

In [76]:
min_support = 0.05  


frequent_itemsets_fp = fpgrowth(encoded_df, min_support=min_support, use_colnames=True)

print("Combos")
print(frequent_itemsets_fp.head())

min_confidence = 0.3
rules_fp = association_rules(frequent_itemsets_fp, metric="confidence", min_threshold=min_confidence)

print("\n(FP-Growth):")
print(rules_fp.head())

d:\Anaconda\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Combos
    support                                           itemsets
0  0.847206                                (time_cat_Not Rush)
1  0.556177                                (crash_type_Single)
2  0.536168  (vehicle_type_involved_No Heavy Vehicle Involved)
3  0.480518                            (speed_limit_Very High)
4  0.429669                                (time_of_day_Night)

(FP-Growth):
           antecedents          consequents  antecedent support  \
0  (holiday_NoHoliday)  (time_cat_Not Rush)            0.962760   
1  (time_cat_Not Rush)  (holiday_NoHoliday)            0.847206   
2  (crash_type_Single)  (time_cat_Not Rush)            0.556177   
3  (time_cat_Not Rush)  (crash_type_Single)            0.847206   
4  (holiday_NoHoliday)  (crash_type_Single)            0.962760   

   consequent support   support  confidence      lift  representativity  \
0            0.847206  0.810036    0.841369  0.993110               1.0   
1            0.962760  0.810036    0.956127  0.9